[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.0 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [17]:
# ✏️ YOUR IMPLEMENTATION HERE

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        self.W_q = nn.Linear(d_model, d_model)
        self.dim = d_model//num_heads
        self.W_k = nn.Linear(d_model, self.dim * num_kv_heads)
        self.W_v = nn.Linear(d_model, self.dim * num_kv_heads)
        self.W_o = nn.Linear(d_model, d_model)
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        # pass  # Initialize projections

    def forward(self, x):
        Q = torch.stack(self.W_q(x).chunk(self.num_heads, dim=-1), dim=1)
        K = torch.stack(self.W_k(x).chunk(self.num_kv_heads, dim=-1), dim=1)
        V = torch.stack(self.W_v(x).chunk(self.num_kv_heads, dim=-1), dim=1)
        K = torch.repeat_interleave(K,self.num_heads//self.num_kv_heads, dim=1)
        V = torch.repeat_interleave(V,self.num_heads//self.num_kv_heads, dim=1)
        logits = torch.einsum('bhid,bhod->bhio', Q, K) / math.sqrt(self.dim)
        attn = torch.softmax(logits, dim=-1)
        output = torch.einsum('bhio,bhod->bhid', attn, V)
        output = output.transpose(1,2).reshape(output.shape[0], output.shape[2], output.shape[1]*output.shape[3])
        output = self.W_o(output)
        return output
        # print(Q.shape, K.shape, V.shape, self.num_heads, len(self.W_q(x).chunk(self.num_heads)))
        pass  # Self-attention with grouped KV

In [10]:
# ✏️ YOUR IMPLEMENTATION HERE

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_model // num_heads
        self.n_rep = num_heads // num_kv_heads
        self.scale = self.head_dim ** -0.5
        self.kv_dim = num_kv_heads * self.head_dim
        self.W_qkv = nn.Linear(d_model, d_model + 2 * self.kv_dim)
        # pass  # Initialize projections

    def forward(self, x):
        B,T,_ = x.shape
        H, G, R, hd = self.num_heads, self.num_kv_heads, self.n_rep, self.head_dim
        qkv = self.W_qkv(x)
        q, k, v = torch.split(qkv, [self.d_model, self.kv_dim, self.kv_dim], dim=-1)

        q = q.view(B, T, G, R, hd).permute(0,2,3,1,4)
        k = k.view(B, T, G, hd).transpose(1,2).unsqueeze(2)
        v = v.view(B, T, G, hd).transpose(1,2).unsqueeze(2)
        logits = torch.einsum('bgroh,bgrih->bgroi', q, k) * self.scale
        attn = torch.softmax(logits, dim=-1)
        out = torch.einsum('bgroi,bgrih->bgroh', attn, v)
        out = out.permute(0,3,1,2,4).reshape(B,T,H * hd)
        return out
        # pass  # Self-attention with grouped KV

In [11]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
# print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
# print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

Output shape: torch.Size([2, 6, 32])


In [12]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (2.3ms)
  💥 [2/5] nn.Linear with correct shapes
     AttributeError: 'GroupQueryAttention' object has no attribute 'W_q'
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<test:nn.Linear with correct shapes>", line 5, in <module>
AttributeError: 'GroupQueryAttention' object has no attribute 'W_q'
  💥 [3/5] Degenerates to MHA when kv_heads == heads
     AttributeError: 'GroupQueryAttention' object has no attribute 'W_k'
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<test:Degenerates to MHA when kv_heads == heads>", line 7, in <module>
AttributeError: 'GroupQueryAttention' object has no attribute 'W_k'
  💥 [4/5] KV heads are shared correctly
     AttributeError: 'GroupQueryAttention' object has no attribute 'W_k'
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  Fil